# 5.7 — ChaseTag 1v1 steered teacher

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/feat/chasetag-steered-teacher/tutorials/5.7_ChaseTag_Steered_Teacher.ipynb)

Play scored `myoChallengeChaseTagFBVs-v0` with `ChaseTagSteeredPair`:
chaser toward the opponent, runner away, both stay inside the ±6 m fence.

This is a **playback** tutorial. The teacher is not a 546 → 354 submission.
To train and submit that file:

| Step | Where |
|---|---|
| Walk in 8 directions (1.0.6) | [5.4](./5.4_MuscleMimic_Directional_Locomotion.ipynb) / [#579](https://github.com/MyoHub/myosuite/pull/579) |
| Export 546 → 354 and submit | [5.6](./5.6_ChaseTag_MyoChallenge.ipynb) |
| Train in the 1v1 game | [#578](https://github.com/MyoHub/myosuite/pull/578) `myoChallengeChaseTagFBVsTrain-v0` |
| PPO action-std cap | [#584](https://github.com/MyoHub/myosuite/pull/584) / [#566](https://github.com/MyoHub/myosuite/issues/566) |
| Expert (obs, act) pairs | `training/chasetag_teacher` |

`amathislab/mm-10m-2` needs musclemimic_models **1.0.5**. The scored observation (546-d) is unchanged.


## Install

In [ ]:
import os

os.environ["MYOSUITE_MUSCLEMIMIC_MODELS_VERSION"] = "1.0.5"
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.pop("MUJOCO_GL", None)

GIT_REF = os.environ.get("MYOSUITE_GIT_REF", "feat/chasetag-steered-teacher")
try:
    import google.colab  # noqa: F401
except ImportError:
    pass
else:
    %pip -q install -U "git+https://github.com/MyoHub/myosuite.git@{GIT_REF}#egg=myosuite[musclemimic]" huggingface_hub imageio

QUICK_MODE = os.environ.get("MYOSUITE_FULL_1V1", "0") != "1"
STEPS = 200 if QUICK_MODE else 2000
print("steps", STEPS)


## Play one episode

In [ ]:
from myosuite import make_env
from myosuite.integrations.musclemimic.steered_teacher import ChaseTagSteeredPair

env = make_env("myoChallengeChaseTagFBVs-v0", render_mode="rgb_array")
raw = env.unwrapped
pair = ChaseTagSteeredPair.load(env)
obs, info = env.reset(seed=0)
pair.reset(raw.model, raw.data, raw._meta, info["task"])
print("task", info["task"])

frames = []
for t in range(STEPS):
    acts = pair.actions(raw.model, raw.data, raw._meta, info["task"], t)
    obs, _r, term, trunc, info = env.step(acts)
    frame = env.render()
    if frame is not None:
        frames.append(frame)
    if term["agent_0"] or trunc["agent_0"]:
        break
print(info["task"], f"{info['elapsed_s']:.2f}s", "tag" if info["tagged"] else info.get("fell"), "score", info["score"])
env.close()


## Video

In [ ]:
from pathlib import Path
import imageio
from IPython.display import Video, display

out = Path("chasetag_teacher.mp4")
if frames:
    with imageio.get_writer(str(out), fps=30, quality=8, macro_block_size=1) as w:
        for fr in frames[::3]:
            w.append_data(fr)
    display(Video(str(out), embed=True))
else:
    print("no frames (headless without a GL backend)")
